In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
  .appName("IcebergLocalDevelopment") \
  .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2") \
  .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
  .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.local.type", "hadoop") \
  .config("spark.sql.catalog.local.warehouse", "spark-warehouse/iceberg") \
  .getOrCreate()

26/10/12 20:49:19 WARN Utils: Your hostname, codespaces-8c3887 resolves to a loopback address: 127.0.0.1; using 10.0.12.41 instead (on interface eth0)
26/10/12 20:49:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/workspaces/learn_iceberg/iceberg_pyspark_project/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/codespace/.ivy2/cache
The jars for the packages stored in: /home/codespace/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-32fced0e-326b-49b2-b385-83312fba75de;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
:: resolution report :: resolve 192ms :: artifacts dl 8ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apa

In [6]:
import os
# Replace this path with the output of 'echo $JAVA_HOME' from your terminal if it differs
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" 

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS schema")



DataFrame[]

In [12]:
spark.sql("""
CREATE TABLE  local.schema.users (
  id INT,
  name STRING,
  age INT
) USING iceberg
""")

DataFrame[]

In [11]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|   schema|
+---------+



In [13]:
# Insert some sample data
spark.sql("""
  INSERT INTO local.schema.users VALUES
    (1, 'Alice', 30),
    (2, 'Bob', 25),
    (3, 'Charlie', 35)""")

DataFrame[]

In [14]:
result = spark.sql("SELECT * FROM local.schema.users")
result.show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 25|
|  3|Charlie| 35|
+---+-------+---+



In [17]:
spark.sql("update local.schema.users set name = 'Harshitha' where id = 2")

DataFrame[]

In [18]:
result = spark.sql("select * from local.schema.users")
result.show()

+---+---------+---+
| id|     name|age|
+---+---------+---+
|  2|Harshitha| 25|
|  3|  Charlie| 35|
|  1|    Alice| 30|
+---+---------+---+



In [19]:
# Delete a record
spark.sql("delete from local.schema.users where id =2")

DataFrame[]

In [22]:
result = spark.sql("SELECT * FROM local.schema.users")
result.show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  3|Charlie| 35|
+---+-------+---+



In [23]:
spark.sql("CREATE table local.schema.users_tgt as Select * from local.schema.users")
spark.sql("Select * from local.schema.users_tgt").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  3|Charlie| 35|
+---+-------+---+



In [25]:
# Merge _tgt table with users table

spark.sql("""
MERGE INTO local.schema.users_tgt AS target
USING local.schema.users AS source
ON target.id = source.id
WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("SELECT * from local.schema.users_tgt").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  3|Charlie| 35|
+---+-------+---+



In [26]:
#Insert a record into users table
spark.sql("INSERT INTO local.schema.users VALUES (2, 'Bob', 25)")

DataFrame[]

In [27]:
spark.sql("SELECT * from local.schema.users").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  2|    Bob| 25|
|  1|  Alice| 30|
|  3|Charlie| 35|
+---+-------+---+



In [28]:
spark.sql("""
MERGE INTO local.schema.users_tgt AS target
USING local.schema.users AS source
ON target.id = source.id
WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("SELECT * from local.schema.users_tgt").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  2|    Bob| 25|
|  1|  Alice| 30|
|  3|Charlie| 35|
+---+-------+---+

